In [ ]:
import os

import torch
import torch.nn.functional as F
from torchvision.transforms import v2
from torchvision import datasets

from snntorch import spikegen

import tonic 
import tonic.transforms as T

from models.spiking_vgg11 import SpikingVGG11
from src.config_utils import setup_config_dictionary
from src.save_utils import load_model
from src.plot_utils import plot_frames, plot_cams, plot_combined_frames
from src.snn_utils import compute_raw_mem, compute_delta_raw_mem
from src.temporal_score_cam import TemporalScoreCAM
from src.data_utils import combine_polarities

config = setup_config_dictionary()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_STEPS = 30

In [ ]:
# load the model
firing_threshold = 1.0
model = SpikingVGG11()

load_model(model, config["MODEL_SAVE_DIR"], model_name="spiking_vgg11", dataset_name="tinyimagenet")

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

tf = v2.Compose([
    v2.Resize(64),
    v2.ToTensor(),
    v2.Normalize(mean, std),
])

DATA_DIR = config["DATA_SAVE_DIR"] + "/tiny-imagenet-200"
TRAIN_DIR = os.path.join(DATA_DIR, "train")

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=tf)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=1, shuffle=False)
# seed for dataloader

# go to seeds for "getting it wrong" -> 4
# torch.manual_seed(12)

model.to(device)
model.eval()

# Visualize using our ScoreCAM implementation
score_cam = TemporalScoreCAM(model, threshold=firing_threshold)
target_layer = None
for module in model.modules():
    if isinstance(module, torch.nn.Conv2d):
        target_layer = module

with torch.no_grad():
    for i, (event_batches, labels) in enumerate(train_loader):
        event_batches = event_batches.to(device)
        labels = labels.to(device)
        event_batches = spikegen.rate(event_batches, num_steps=NUM_STEPS)  # Convert images to spikes, # T, B, 3, 64 ,64
        event_batches = event_batches.permute(1, 0, 2, 3, 4)  # B, T, C, H, W
        out_spikes, out_mem = model(event_batches)
        raw_mem = compute_raw_mem(out_spikes, out_mem, firing_threshold)
        delta_raw_mem = compute_delta_raw_mem(raw_mem)
        output_class = F.softmax(out_spikes.sum(dim=1), dim=1).cpu().numpy()
        predicted_label = int(output_class.argmax(axis=1)[0])
        print(f"Output probabilities: {[f'{p:.2%}' for p in output_class[0]]}\nFrom spike sum {out_spikes.sum(dim=1)}")
        print(f"Predicted label {predicted_label},real label: {labels.item()}\n")
        combined_frames = combine_polarities(event_batches[0].cpu().numpy())
        # Compute ScoreCAM
        cam = score_cam.generate_cam(event_batches, target_layer=target_layer)
        plot_frames(event_batches[0].cpu().numpy())
        plot_combined_frames(combined_frames)
        print("Cams for predicted label:")
        plot_cams(cam, class_idx=predicted_label)
        print("Cams for real label:")
        plot_cams(cam, class_idx=labels.item())
        print(f"Cams for random label, e.g. {0}:")
        plot_cams(cam, class_idx=0)

        # only one sample for now
        break